In [1]:
import json
import time
import openai
from pathlib import Path
from tqdm import tqdm




In [2]:
# === Config ===
INPUT_FILE = "../data/books/breath_semantic_chunks.jsonl"
OUTPUT_FILE = "../data/books/breath_gpt4o_finetune_data.jsonl"
MODEL = "gpt-4o"
RATE_LIMIT_SECONDS = 1.2  # To avoid rate limit issues



In [ ]:
# Optional: set your API key here or use the environment variable
# openai.api_key = "sk-..."

def generate_prompt(chunk_text):
    """Creates the prompt to send to GPT-4o."""
    return (
        f"Based on the following text from a Buddhist meditation manual, generate a guided meditation script "
        f"that preserves the tone and teachings of the original. Keep it in second person (you), clear, and calm.\n\n"
        f"Source:\n{chunk_text}\n\nGuided Meditation:"
    )

def call_openai(prompt):
    """Calls the OpenAI API with the given prompt."""
    try:
        response = openai.ChatCompletion.create(
            model=MODEL,
            messages=[
                {"role": "system", "content": "You are a skilled meditation teacher."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=800  # Adjust as needed
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"Error: {e}")
        return None

def main():
    with open(INPUT_FILE, "r", encoding="utf-8") as f:
        chunks = [json.loads(line) for line in f]

    with open(OUTPUT_FILE, "w", encoding="utf-8") as out_f:
        for i, chunk in enumerate(tqdm(chunks, desc="Processing chunks")):
            prompt = generate_prompt(chunk["text"])
            output = call_openai(prompt)

            if output:
                json.dump({
                    "instruction": "Generate a guided meditation from a Buddhist text excerpt.",
                    "input": chunk["text"],
                    "output": output
                }, out_f, ensure_ascii=False)
                out_f.write("\n")
            else:
                print(f"Skipping chunk {i} due to error.")

            time.sleep(RATE_LIMIT_SECONDS)

    print(f"\nSaved fine-tune data to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()